# Clase 9 — Práctica: VQE fundamentos

Vas a comprobar en carne propia por qué el ansatz importa tanto como el Hamiltoniano: un circuito que funciona perfectamente para un Hamiltoniano puede quedarse atascado con otro, si no le das suficiente libertad de giro.

## Enunciado

En el primer ejemplo de la clase teórica (Hamiltoniano de un único qubit) usaste `H = 1 + 2 * cirq.Z(qubit)` con un ansatz de una sola rotación `ry`, y el optimizador encontró la energía mínima teórica (−1) sin problema.

Ahora cambia el Hamiltoniano a `H = 1 + 2 * cirq.X(qubit)`. Antes de ejecutar nada, piensa: el ansatz actual solo tiene una rotación `ry`, que nunca puede llevar el qubit a un autoestado de X. Si lo ejecutas tal cual, el optimizador se queda atascado lejos del mínimo real.

Tu tarea:

1. Añade una puerta `cirq.rz` (con un segundo parámetro entrenable) al ansatz, para que el circuito tenga suficiente libertad de giro hacia cualquier eje
2. Actualiza el Hamiltoniano a `1 + 2 * cirq.X(qubit)`
3. Vuelve a optimizar con Nelder-Mead y comprueba que ahora sí converge a la energía mínima teórica (−1)

**Pistas**

- Necesitas dos símbolos de `sympy` en vez de uno, por ejemplo `sympy.symbols("angulo_y angulo_z")`
- El orden de las puertas en el ansatz importa: aplica primero `cirq.ry` y después `cirq.rz` sobre el mismo qubit
- La función `energia(parametros)` recibe ahora una lista de dos valores, no de uno; `np.random.uniform(0, 2 * np.pi, 2)` te da un punto de partida aleatorio con dos parámetros
- La energía mínima teórica de `H = 1 + 2*X` es la misma que la de `H = 1 + 2*Z` (−1), porque X y Z tienen los mismos autovalores (+1 y −1); lo único que cambia es el eje hacia el que tiene que girar el ansatz

In [ ]:
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
from scipy.optimize import minimize

qubit = cirq.GridQubit(0, 0)

# Ansatz original: una única rotación Y entrenable (insuficiente para H = 1 + 2*X)
angulo = sympy.symbols("angulo")
circuito_vqe = cirq.Circuit(cirq.ry(angulo).on(qubit))

# TODO: cambia el Hamiltoniano de "1 + 2 * cirq.Z(qubit)" a "1 + 2 * cirq.X(qubit)"
hamiltoniano = 1 + 2 * cirq.Z(qubit)

# TODO: añade un segundo símbolo entrenable (por ejemplo, "angulo_z") y reconstruye
# el circuito para que incluya, además de la rotación ry, una rotación rz sobre
# el mismo qubit

entrada_vacia = tfq.convert_to_tensor([cirq.Circuit()])
capa_entrada = tf.keras.layers.Input(shape=(), dtype=tf.dtypes.string)
inicializador = tf.keras.initializers.RandomUniform(0, 2 * np.pi)
salida = tfq.layers.PQC(circuito_vqe, hamiltoniano,
                         differentiator=tfq.differentiators.CentralDifference(),
                         initializer=inicializador)(capa_entrada)
modelo_vqe = tf.keras.models.Model(inputs=capa_entrada, outputs=salida)


def energia(parametros):
    modelo_vqe.set_weights(np.array([parametros]))
    return modelo_vqe(entrada_vacia).numpy()[0][0]


# TODO: el punto de partida aleatorio ahora necesita tantos valores como
# parámetros entrenables tenga tu ansatz (uno por cada símbolo de sympy)
resultado = minimize(energia, np.random.uniform(0, 2 * np.pi, 1), method="Nelder-Mead")
print("Energía mínima encontrada:", resultado["fun"])
print("Parámetros óptimos:", resultado["x"])

<details>
<summary>💡 Ver solución</summary>

```python
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
from scipy.optimize import minimize

qubit = cirq.GridQubit(0, 0)

# Ansatz con dos parámetros entrenables: primero ry, después rz.
# La rz añade la libertad de giro que faltaba para alcanzar un autoestado de X
angulo_y, angulo_z = sympy.symbols("angulo_y angulo_z")
circuito_vqe = cirq.Circuit(
    cirq.ry(angulo_y).on(qubit),
    cirq.rz(angulo_z).on(qubit),
)

# Hamiltoniano cambiado de Z a X: H = 1*I + 2*X, energía fundamental teórica -1
hamiltoniano = 1 + 2 * cirq.X(qubit)

entrada_vacia = tfq.convert_to_tensor([cirq.Circuit()])
capa_entrada = tf.keras.layers.Input(shape=(), dtype=tf.dtypes.string)
inicializador = tf.keras.initializers.RandomUniform(0, 2 * np.pi)
salida = tfq.layers.PQC(circuito_vqe, hamiltoniano,
                         differentiator=tfq.differentiators.CentralDifference(),
                         initializer=inicializador)(capa_entrada)
modelo_vqe = tf.keras.models.Model(inputs=capa_entrada, outputs=salida)


def energia(parametros):
    modelo_vqe.set_weights(np.array([parametros]))
    return modelo_vqe(entrada_vacia).numpy()[0][0]


# Punto de partida aleatorio con dos parámetros, uno por cada símbolo del ansatz
resultado = minimize(energia, np.random.uniform(0, 2 * np.pi, 2), method="Nelder-Mead")
print("Energía mínima encontrada:", resultado["fun"])
print("Parámetros óptimos:", resultado["x"])

# Con la rz añadida, el resultado converge a -1 (o muy cerca), la misma energía
# fundamental teórica que en el ejemplo original con Z, porque X y Z comparten
# los mismos autovalores (+1 y -1); lo que cambiaba era el eje de giro necesario
```

</details>